# 8장 이야기 데이터를 숫자 표로: 텐서와 임베딩

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 8-1 어린이 이야기 데이터

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from torch import nn
from modu_llm import load_stories, Vocab, encode_texts, show

stories = load_stories(200)
print("이야기 수:", len(stories))
print(stories[0][:100])

## 실습 8-2 글자 사전

In [ ]:
vocab = Vocab.build()
data = encode_texts(stories, vocab)
print("사전 크기:", len(vocab))
print("글자 수:", len(data))
print("처음 10글자:", data[:10])
print("되돌리기:", vocab.decode(data[:10]))

## 실습 8-3 글자 번호를 텐서로

In [ ]:
sentence = "토끼가 숲에 가요"
ids = torch.tensor(vocab.encode(sentence))
print(ids.dtype)

chars = list(sentence.replace(" ", "␣"))
show(ids, cols=chars)

## 실습 8-4 텐서의 모양

In [ ]:
one = torch.tensor(3.5)                           # 숫자 하나
card = torch.tensor([1.0, 0.0, 2.0, 0.0])         # 숫자 카드 한 장
cards = torch.tensor([[1.0, 0.0, 2.0, 0.0],       # "토"의 숫자 카드
                      [0.0, 1.0, 0.0, 1.0],       # "끼"의 숫자 카드
                      [2.0, 0.0, 0.0, 1.0]])      # "가"의 숫자 카드
print(one.shape)
print(card.shape)
print(cards.shape)
print(ids.shape)

names = ["토", "끼", "가"]
show(cards, rows=names)

## 실습 8-5 줄, 칸, 세로줄 꺼내기

**먼저 예상해 보세요.** cards[:, 0]에는 어떤 숫자 세 개가 나올까요?

In [ ]:
show(cards[0])                  # 0번 줄: "토"의 숫자 카드
show(cards[2, 3])               # 2번 줄의 3번 칸
show(cards[:, 0], cols=names)   # 모든 줄의 0번 칸

## 실습 8-6 쌓는 모양 바꾸기: view

In [ ]:
print(ids.shape, "→", ids.view(3, 3).shape)
show(ids, cols=chars)
show(ids.view(3, 3))    # 번호 9개를 3줄 × 3칸으로

## 실습 8-7 번호로 숫자 카드 꺼내기

**먼저 예상해 보세요.** "토끼가 숲에 가요"에는 '가'가 두 번 나옵니다. 두 '가'의 숫자 카드는 같을까요?

In [ ]:
torch.manual_seed(0)
emb = nn.Embedding(len(vocab), 8)
print("표 모양:", emb.weight.shape)

cards9 = emb(ids)
print("숫자 카드 모양:", cards9.shape)
print("표의 줄을 꺼낸 것과 같은가요?", torch.equal(cards9[0], emb.weight[ids[0]]))
print("두 '가'의 카드가 같은가요?", torch.equal(cards9[2], cards9[7]))
show(cards9, rows=chars, mark_rows=[2, 7])